# kl-divergence-gaussian-closed-form — worked example 2: Three-stage reduction: per-element → per-sample → batch mean

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kl-divergence-gaussian-closed-form`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The VAE KL loss proceeds in three stages. First compute per-element contributions using the closed form. Then sum across the latent dimension to get a scalar per sample (since a diagonal Gaussian factors across dimensions). Finally take the mean across the batch. Each stage has a distinct shape: `(B, D)` → `(B,)` → scalar.

## Worked solution

Step 1: Compute `per_elem = -0.5 * (1 + 2*logsigma - mu^2 - exp(2*logsigma))` — the ARENA textbook sign convention (equivalent to the other form).

Step 2: `per_sample = per_elem.sum(dim=1)` collapses the `D` latent dimension. Each entry is the KL between the encoder's Gaussian for that sample and `N(0, I)`.

Step 3: `scalar = per_sample.mean()` averages across the batch. This is the KL term added to the reconstruction loss in the ELBO.

Step 4: Print all three tensors and their shapes to confirm the reduction sequence.

In [ ]:
import torch as t

t.manual_seed(1)

def kl_gaussian_three_stages(mu: t.Tensor, logsigma: t.Tensor):
    B, D = mu.shape
    # Stage 1: per-element, shape (B, D)
    per_elem = -0.5 * (1 + 2 * logsigma - mu.pow(2) - (2 * logsigma).exp())
    # Stage 2: sum over latent dim, shape (B,)
    per_sample = per_elem.sum(dim=1)
    # Stage 3: mean over batch, scalar
    scalar = per_sample.mean()
    return per_elem, per_sample, scalar

B, D = 4, 5
mu = t.randn(B, D) * 0.5
logsigma = t.randn(B, D) * 0.3

pe, ps, sc = kl_gaussian_three_stages(mu, logsigma)
print(f'per_elem  shape: {pe.shape}')   # (4, 5)
print(f'per_sample shape: {ps.shape}')  # (4,)
print(f'scalar    shape: {sc.shape}')   # ()
print(f'per_sample values: {ps.tolist()}')
print(f'batch mean KL: {sc.item():.4f}')

# All per-sample KLs must be >= 0 (KL is non-negative)
for i, v in enumerate(ps.tolist()):
    assert v >= -1e-5, f'Sample {i} has negative KL: {v}'
print('All KLs non-negative — OK.')